#Запросы к базе данных


## Подготовка к работе

In [15]:
!wget -O cian.db https://raw.githubusercontent.com/menneli/PDA-1/main/Lab1/cian.db

--2026-10-10 11:54:57--  https://raw.githubusercontent.com/menneli/PDA-1/main/Lab1/cian.db
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 647168 (632K) [application/octet-stream]
Saving to: ‘cian.db’

cian.db             100%[===================>] 632.00K  --.-KB/s    in 0.02s   

2026-10-10 11:54:58 (35.1 MB/s) - ‘cian.db’ saved [647168/647168]



In [37]:
import pandas as pd
from sqlalchemy import create_engine, MetaData, Table

engine = create_engine("sqlite:///cian.db")
metadata = MetaData()
cian = Table("listings", metadata, autoload_with=engine)


## Запросы с sqlalchemy core

1) Средняя цена за квартиру на каждой станции метро

In [54]:
from sqlalchemy import desc, func, select, text
query_1 = (
    select(
        cian.c.underground,
        func.round(func.avg(cian.c.price),0).label("avg_price")
    ).group_by(cian.c.underground).order_by(desc("avg_price"))
)
with engine.connect() as conn:
  result_1 = conn.execute(query_1).fetchall()


# Для более приятного вывода
df = pd.DataFrame(result_1)
df


,underground,avg_price
0,Звенигородская,198000000.0
1,Крестовский остров,109027423.0
2,Спортивная,96612368.0
3,Чернышевская,78384959.0
4,Площадь Ленина,77530000.0
...,...,...
66,Парнас,11051000.0
67,Девяткино,10266484.0
68,Рыбацкое,10112143.0
69,Шушары,9413833.0


2) Вывести информацию по квартирам дешевле, чем средняя по рынку

In [55]:
from sqlalchemy import desc, func, select, text

#подзапрос для основого
avg_price = (select(func.avg(cian.c.price)).scalar_subquery())

query_2 = (
    select(
        cian.c
    ).where(cian.c.price < avg_price).order_by(cian.c.price)
)
with engine.connect() as conn:
  result_2 = conn.execute(query_2).fetchall()

#Для приятного вывода
df = pd.DataFrame(result_2)
df

,id,url,author,author_type,location,deal_type,accommodation_type,floor,floors_count,rooms_count,total_meters,price,price_per_m2,district,street,house_number,underground,residential_complex,is_active
0,334715187,https://spb.cian.ru/sale/flat/334715187/,МЭТС,real_estate_agent,Санкт-Петербург,sale,flat,2,4,1,20.40,4086400,200313.725490,Красносельский,Калинина,10,Юго-Западная,None,1
1,334033846,https://spb.cian.ru/sale/flat/334033846/,Невский простор,real_estate_agent,Санкт-Петербург,sale,flat,1,5,1,14.77,4200000,284360.189573,Центральный,Маяковского,19/15,Чернышевская,None,1
2,334159058,https://spb.cian.ru/sale/flat/334159058/,Гарант-Сервис,real_estate_agent,Санкт-Петербург,sale,flat,5,5,2,44.20,5270000,119230.769231,Колпинский,Южная,33,Рыбацкое,None,1
3,334509897,https://spb.cian.ru/sale/flat/334509897/,Гарант-Сервис,real_estate_agent,Санкт-Петербург,sale,flat,1,5,3,41.50,5550000,133734.939759,Колпинский,Пролетарская,83,Шушары,None,1
4,332264003,https://spb.cian.ru/sale/flat/332264003/,Колвэй,real_estate_agent,Санкт-Петербург,sale,flat,5,5,3,42.20,5750000,136255.924171,Колпинский,Танкистов,30,Шушары,None,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
982,332407328,https://spb.cian.ru/sale/flat/332407328/,ЛСР Санкт-Петербург,developer,Санкт-Петербург,sale,flat,2,8,2,74.20,36001840,485200.000000,Петроградский,Ремесленная,15,Спортивная,NEVA RESIDENCE,1
983,334447624,https://spb.cian.ru/sale/flat/334447624/,Fizika Development,developer,Санкт-Петербург,sale,flat,5,10,2,71.77,36127797,503382.987321,Адмиралтейский,Перевозная,None,Садовая,Остров первых,1
984,333356466,https://spb.cian.ru/sale/flat/333356466/,Этажи Санкт-Петербург,real_estate_agent,Санкт-Петербург,sale,flat,13,19,3,108.20,36600000,338262.476895,Московский,Малая Митрофаньевская,8к1,Московские ворота,Галактика. Премиум,1
985,318116064,https://spb.cian.ru/sale/flat/318116064/,Балтийская Коммерция,real_estate_agent,Санкт-Петербург,sale,flat,2,9,2,86.30,36677500,425000.000000,Петроградский,Чапаева,17к2,Петроградская,Мануфактура,1


##  Запросы  pandas (непосредтсвенно или через делегирование запросов)

3) Средняя площадь по количеству комнат

In [59]:
data = pd.read_sql(text("SELECT * FROM listings"), engine)
result_3 = data.groupby("rooms_count")["total_meters"].mean().reset_index(name="average_area")
result_3

,rooms_count,average_area
0,-1,77.766667
1,1,40.594220
2,2,65.355320
3,3,94.519865
4,4,139.947459
5,5,198.939535


-1 в данном случае значит доля в квартире
(Примечание автора)

4. Медианная цена квартиры по количеству комнат

In [60]:
query_4 = """
SELECT rooms_count, price FROM listings;
"""
df = pd.read_sql(query_4, engine)
result_4 = (df.groupby("rooms_count")["price"].median().reset_index())
result_4

,rooms_count,price
0,-1,10500000.0
1,1,12777500.0
2,2,20600000.0
3,3,30200000.0
4,4,58000000.0
5,5,58000000.0


-1 в данном случае значит доля в квартире
(Примечание автора)

5) Авторы с самым большим количеством объектов

In [61]:
query_5 = """SELECT author, COUNT(id) as total_count
FROM listings
GROUP BY author
ORDER BY total_count DESC
LIMIT 5
"""
result_5 = pd.read_sql(query_5, engine)

df = pd.DataFrame(result_5)
df

,author,total_count
0,Этажи Санкт-Петербург,225
1,Мир Квартир Элит,86
2,ЛСР Санкт-Петербург,43
3,ELEMENT,43
4,Роман Корякин,38
